In [7]:
import numpy as np
from itertools import combinations

# ============================================================
# PARAMETERS
# ============================================================
machine_names = ['CNC','Comp','IMM','Conv','EAF','HVAC','Weld','Pump','Hyd','Motor']
machine_full  = ['CNC Milling','Air Compressor','Injection Molder','Conveyor Belt',
                 'Elec Arc Furnace','HVAC/Chiller','Robotic Welder','Water Pump',
                 'Hydraulic Press','Motor Drive']
P      = [22.4, 22.0, 45.0, 3.7, 500.0, 76.0, 12.0, 18.5, 22.0, 30.0]
R      = 7.48
D_rate = 235.15
alpha  = D_rate / (30 * 32)
K      = 8
lam    = 20000
n      = len(P)

# ============================================================
# COEFFICIENTS
# ============================================================
c = [R*P[i]*0.25 + alpha*P[i]**2 for i in range(n)]
q = {}
for i in range(n):
    for j in range(i+1, n):
        q[(i,j)] = 2 * alpha * P[i] * P[j]

def compute_cost(x):
    cost  = sum(c[i]*x[i] for i in range(n))
    cost += sum(q[(i,j)]*x[i]*x[j] for i in range(n) for j in range(i+1,n))
    return cost

def compute_qubo_energy(x):
    energy  = sum((c[i] + lam*(1-2*K))*x[i] for i in range(n))
    energy += sum((q[(i,j)] + 2*lam)*x[i]*x[j] for i in range(n) for j in range(i+1,n))
    return energy

# ============================================================
# BASELINE
# ============================================================
x_base = [1]*n
f_base = compute_cost(x_base)

# ============================================================
# BRUTE FORCE
# ============================================================
best_brute_cost = float('inf')
best_brute_combo = None
for r in range(K, n+1):
    for combo in combinations(range(n), r):
        x = [1 if i in combo else 0 for i in range(n)]
        cost = compute_cost(x)
        if cost < best_brute_cost:
            best_brute_cost = cost
            best_brute_combo = list(combo)

# ============================================================
# SIMULATED ANNEALING
# ============================================================
def simulated_annealing(seed=42):
    rng = np.random.default_rng(seed)
    x = [1]*n
    energy = compute_qubo_energy(x)
    best_x = x[:]
    best_energy = energy
    T_start = 10000.0
    T_end   = 0.01
    n_sweeps = 10000
    T    = T_start
    cool = (T_end / T_start) ** (1.0 / n_sweeps)
    history = []
    for sweep in range(n_sweeps):
        i = int(rng.integers(0, n))
        x_new    = x[:]
        x_new[i] = 1 - x_new[i]
        energy_new = compute_qubo_energy(x_new)
        delta = energy_new - energy
        if delta < 0 or rng.random() < np.exp(-delta / T):
            x, energy = x_new, energy_new
            if energy < best_energy:
                best_energy = energy
                best_x = x[:]
        T *= cool
        if sweep % 200 == 0:
            history.append((sweep, compute_cost(best_x)))
    return best_x, history

# Run 30 seeds
run_costs = []
best_qisa_x, best_qisa_cost = None, float('inf')
all_histories = []
for seed in range(30):
    x_sol, hist = simulated_annealing(seed=seed)
    cost = compute_cost(x_sol)
    run_costs.append(cost)
    all_histories.append(hist)
    if cost < best_qisa_cost:
        best_qisa_cost, best_qisa_x = cost, x_sol[:]

active_idx   = [i for i in range(n) if best_qisa_x[i]==1]
removed_idx  = [i for i in range(n) if best_qisa_x[i]==0]
savings      = f_base - best_qisa_cost
gap_pct      = abs(best_qisa_cost - best_brute_cost) / best_brute_cost * 100
conv_sweeps  = [h[0] for h in all_histories[0]]
conv_costs   = [h[1] for h in all_histories[0]]

# ============================================================
# REPORT — DISPLAY
# ============================================================
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Header ──────────────────────────────────────────────────────
display(HTML("""
<div style="border-bottom:2px solid #333;padding-bottom:12px;margin-bottom:20px;">
  <p style="font-size:11px;color:#888;margin:0;letter-spacing:2px;text-transform:uppercase;">
    NEOZEP Research · BSc Industrial Engineering · Introduction to Metaheuristics
  </p>
  <h2 style="margin:4px 0 0;font-size:18px;color:#111;">
    QUBO-Based Quantum-Inspired Simulated Annealing
  </h2>
  <p style="margin:2px 0 0;font-size:13px;color:#555;">
    Demand Charge Optimization · 10 Machines · K = 8 · λ = 20,000 · 30 Seeds
  </p>
</div>
"""))

# ── Metric cards ────────────────────────────────────────────────
display(HTML(f"""
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:20px;">
  <div style="border:1px solid #ddd;border-radius:8px;padding:14px;text-align:center;">
    <div style="font-size:11px;color:#888;margin-bottom:4px;">Baseline cost</div>
    <div style="font-size:18px;font-weight:600;color:#c0392b;">PHP {f_base:,.2f}</div>
    <div style="font-size:11px;color:#aaa;margin-top:4px;">All 10 machines ON</div>
  </div>
  <div style="border:1px solid #ddd;border-radius:8px;padding:14px;text-align:center;">
    <div style="font-size:11px;color:#888;margin-bottom:4px;">Optimal cost</div>
    <div style="font-size:18px;font-weight:600;color:#27ae60;">PHP {best_qisa_cost:,.2f}</div>
    <div style="font-size:11px;color:#aaa;margin-top:4px;">QI-SA / Brute force</div>
  </div>
  <div style="border:1px solid #ddd;border-radius:8px;padding:14px;text-align:center;">
    <div style="font-size:11px;color:#888;margin-bottom:4px;">Savings</div>
    <div style="font-size:18px;font-weight:600;color:#27ae60;">PHP {savings:,.2f}</div>
    <div style="font-size:11px;color:#aaa;margin-top:4px;">{savings/f_base*100:.2f}% reduction</div>
  </div>
  <div style="border:1px solid #ddd;border-radius:8px;padding:14px;text-align:center;">
    <div style="font-size:11px;color:#888;margin-bottom:4px;">Optimality gap</div>
    <div style="font-size:18px;font-weight:600;color:#27ae60;">{gap_pct:.4f}%</div>
    <div style="font-size:11px;color:#aaa;margin-top:4px;">QI-SA = brute force</div>
  </div>
</div>
"""))

# ── Chart 1: Cost comparison ─────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=['Baseline<br>(all 10 ON)', 'Brute Force<br>Optimal', 'QI-SA<br>Result'],
    y=[f_base, best_brute_cost, best_qisa_cost],
    marker_color=['#e74c3c','#27ae60','#2980b9'],
    text=[f'PHP {v:,.2f}' for v in [f_base, best_brute_cost, best_qisa_cost]],
    textposition='outside', textfont_size=11
))
fig1.update_layout(
    title=dict(text='Cost Comparison (PHP/period)', font_size=14, x=0.5),
    yaxis=dict(title='Cost (PHP)', tickformat=',.0f', gridcolor='#eee'),
    plot_bgcolor='white', paper_bgcolor='white',
    height=320, margin=dict(t=50,b=40,l=70,r=20),
    showlegend=False
)
fig1.show()

# ── Chart 2: Machine cost bar ────────────────────────────────────
colors_bar = ['#27ae60' if i in active_idx else '#e74c3c' for i in range(n)]
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=machine_names, y=c,
    marker_color=colors_bar,
    text=[f'{v:,.0f}' for v in c],
    textposition='outside', textfont_size=10
))
fig2.update_layout(
    title=dict(text='Linear Cost c_i per Machine (PHP) — green: active · red: removed', font_size=13, x=0.5),
    yaxis=dict(title='c_i (PHP)', tickformat=',.0f', gridcolor='#eee'),
    plot_bgcolor='white', paper_bgcolor='white',
    height=320, margin=dict(t=50,b=40,l=70,r=20),
    showlegend=False
)
fig2.show()

# ── Chart 3: 10x10 heatmap ───────────────────────────────────────
import numpy as np_heat
Q_heat = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        Q_heat[i][j] = q[(i,j)]
        Q_heat[j][i] = q[(i,j)]
text_heat = [[f'{Q_heat[i][j]:,.0f}' if i != j else '—' for j in range(n)] for i in range(n)]
fig3 = go.Figure(data=go.Heatmap(
    z=Q_heat, x=machine_names, y=machine_names,
    colorscale='Reds',
    text=text_heat, texttemplate='%{text}', textfont_size=9,
    colorbar=dict(title='PHP')
))
fig3.update_layout(
    title=dict(text='Pairwise Demand Interaction Coefficients q_ij (PHP/period)', font_size=13, x=0.5),
    height=420, margin=dict(t=60,b=40,l=80,r=20)
)
fig3.show()

# ── Chart 4: 30-seed convergence ────────────────────────────────
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=list(range(30)), y=run_costs,
    mode='markers+lines',
    marker=dict(color='#2980b9', size=8),
    line=dict(color='#2980b9', width=1.5),
))
fig4.add_hline(
    y=best_brute_cost, line_dash='dash', line_color='#27ae60',
    annotation_text=f'Brute force: PHP {best_brute_cost:,.2f}',
    annotation_position='top right'
)
fig4.update_layout(
    title=dict(text='QI-SA Cost per Seed (30 Independent Runs)', font_size=13, x=0.5),
    xaxis=dict(title='Seed'),
    yaxis=dict(title='Cost (PHP)', tickformat=',.0f', gridcolor='#eee'),
    plot_bgcolor='white', paper_bgcolor='white',
    height=300, margin=dict(t=50,b=40,l=70,r=20),
    showlegend=False
)
fig4.show()

# ── Chart 5: SA convergence ──────────────────────────────────────
fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=conv_sweeps, y=conv_costs,
    mode='lines', line=dict(color='#8e44ad', width=2),
    fill='tozeroy', fillcolor='rgba(142,68,173,0.08)'
))
fig5.update_layout(
    title=dict(text='SA Energy Convergence — Seed 0 (cost vs sweep)', font_size=13, x=0.5),
    xaxis=dict(title='Sweep'),
    yaxis=dict(title='Best cost so far (PHP)', tickformat=',.0f', gridcolor='#eee'),
    plot_bgcolor='white', paper_bgcolor='white',
    height=280, margin=dict(t=50,b=40,l=70,r=20),
    showlegend=False
)
fig5.show()

# ── Machine selection table ──────────────────────────────────────
rows = ""
for i in range(n):
    is_active = i in active_idx
    bg     = '#000080' if is_active else '#87CEEB'
    status = '✅ Active'  if is_active else '❌ Removed'
    note   = {4:'Highest c_i — dominant demand cost', 5:'Removed with EAF; high interaction cost'}.get(i, 'Optimal subset member') if not is_active else 'Included in optimal subset'
    rows += f"""<tr style="background:{bg};">
      <td style="padding:7px 12px;">{machine_full[i]}</td>
      <td style="padding:7px 12px;text-align:right;">{P[i]}</td>
      <td style="padding:7px 12px;text-align:right;">{c[i]:,.2f}</td>
      <td style="padding:7px 12px;">{status}</td>
      <td style="padding:7px 12px;font-size:12px;color:#666;">{note}</td>
    </tr>"""
display(HTML(f"""
<h4 style="font-size:13px;font-weight:600;margin:16px 0 8px;">
  Machine Selection Result &nbsp;
  <span style="background:#e8f5e9;color:#2e7d32;font-size:11px;padding:3px 8px;border-radius:4px;">
    {sum(best_qisa_x)} active · {n - sum(best_qisa_x)} removed · K={K} satisfied
  </span>
</h4>
<table style="width:100%;border-collapse:collapse;font-size:13px;">
  <thead>
    <tr style="background:#2c3e50;color:white;">
      <th style="padding:8px 12px;text-align:left;">Machine</th>
      <th style="padding:8px 12px;text-align:right;">P_i (kW)</th>
      <th style="padding:8px 12px;text-align:right;">c_i (PHP)</th>
      <th style="padding:8px 12px;text-align:left;">Status</th>
      <th style="padding:8px 12px;text-align:left;">Note</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
"""))

# ── Summary block ────────────────────────────────────────────────
removed_names = [machine_full[i] for i in removed_idx]
active_names  = [machine_full[i] for i in active_idx]
display(HTML(f"""
<div style="background:#000080;border-left:4px solid #27ae60;padding:14px 18px;
            border-radius:0 8px 8px 0;margin-top:16px;font-family:monospace;font-size:13px;">
  <strong>Final Results</strong><br><br>
  Baseline (all ON):&nbsp;&nbsp;&nbsp; PHP {f_base:>14,.4f}<br>
  Brute force optimal:&nbsp; PHP {best_brute_cost:>14,.4f}<br>
  QI-SA best result:&nbsp;&nbsp;&nbsp; PHP {best_qisa_cost:>14,.4f}<br>
  Savings:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; PHP {savings:>14,.4f} &nbsp;({savings/f_base*100:.2f}%)<br>
  Optimality gap:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {gap_pct:.4f}%<br>
  Std deviation (30 runs): PHP {np.std(run_costs):>14,.4f}<br>
  Removed:&nbsp;&nbsp; {', '.join(removed_names)}<br>
  Active:&nbsp;&nbsp;&nbsp; {', '.join(active_names)}
</div>
"""))

print("\nReport complete.")

Machine,P_i (kW),c_i (PHP),Status,Note
CNC Milling,22.4,164.79,✅ Active,Included in optimal subset
Air Compressor,22.0,159.69,✅ Active,Included in optimal subset
Injection Molder,45.0,580.17,✅ Active,Included in optimal subset
Conveyor Belt,3.7,10.27,✅ Active,Included in optimal subset
Elec Arc Furnace,500.0,"62,171.98",❌ Removed,Highest c_i — dominant demand cost
HVAC/Chiller,76.0,"1,556.94",❌ Removed,Removed with EAF; high interaction cost
Robotic Welder,12.0,57.71,✅ Active,Included in optimal subset
Water Pump,18.5,118.43,✅ Active,Included in optimal subset
Hydraulic Press,22.0,159.69,✅ Active,Included in optimal subset
Motor Drive,30.0,276.55,✅ Active,Included in optimal subset



Report complete.
